In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# K-Means Clustering: Diabetes Readmission Analysis

프로젝트 계획서와 데이터 파일은 UCI **Diabetes 130-US hospitals for years 1999-2008** 데이터를 바탕으로 합니다. 이 데이터는 당뇨병 환자의 병원 방문 기록, 입원/퇴원 정보, 검사 및 투약 정보, 진단 코드, 재입원 여부(`readmitted`)를 포함합니다.

예측/분석 목표는 환자의 임상 및 병원 이용 패턴을 기반으로 **재입원 위험이 높은 환자군을 파악**하는 것입니다. K-Means는 지도학습 분류 모델이 아니라 비지도 군집화 알고리즘이므로, 이 노트북에서는 `readmitted`를 군집 생성에는 사용하지 않고, 군집 생성 후 각 군집의 재입원율을 비교하여 위험군을 해석합니다.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans

RANDOM_STATE = 42
DATA_PATH = '/content/drive/MyDrive/SKKU/M2_1/diabetic_data.csv'

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv(DATA_PATH)

print('Shape:', df.shape)
display(df.head())
display(df.info())

In [ ]:
df = df.replace('?', np.nan)

target_counts = df['readmitted'].value_counts(dropna=False)
missing_rate = df.isna().mean().sort_values(ascending=False).head(15)

print('readmitted distribution')
display(target_counts)

print('\nTop missing-rate columns')
display((missing_rate * 100).round(2).to_frame('missing_percent'))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='readmitted', order=['NO', '>30', '<30'])
plt.title('Readmission Target Distribution')
plt.xlabel('readmitted')
plt.ylabel('count')
plt.show()

## Feature Engineering Strategy

- `readmitted`는 원래 `NO`, `>30`, `<30` 세 값입니다.
- 재입원 위험 분석을 위해 `NO=0`, `>30` 또는 `<30=1`인 이진 타깃 `readmitted_binary`를 만듭니다.
- K-Means 학습에는 `readmitted`와 `readmitted_binary`를 넣지 않습니다.
- 결측이 매우 많은 `weight`, `payer_code`, 그리고 식별자인 `encounter_id`, `patient_nbr`는 제외합니다.
- ICD 진단 코드는 큰 질병군으로 단순화해 희소한 범주 수를 줄입니다.

In [ ]:
def age_to_midpoint(age_value):
    if pd.isna(age_value):
        return np.nan
    left, right = age_value.strip('[]()').split('-')
    return (int(left) + int(right)) / 2

def diagnosis_group(code):
    if pd.isna(code):
        return 'Missing'
    try:
        value = float(code)
    except ValueError:
        return 'Other'

    if 390 <= value <= 459 or value == 785:
        return 'Circulatory'
    if 460 <= value <= 519 or value == 786:
        return 'Respiratory'
    if 520 <= value <= 579 or value == 787:
        return 'Digestive'
    if 250 <= value < 251:
        return 'Diabetes'
    if 800 <= value <= 999:
        return 'Injury'
    if 710 <= value <= 739:
        return 'Musculoskeletal'
    if 580 <= value <= 629 or value == 788:
        return 'Genitourinary'
    if 140 <= value <= 239:
        return 'Neoplasms'
    return 'Other'

model_df = df.copy()
model_df['readmitted_binary'] = model_df['readmitted'].isin(['<30', '>30']).astype(int)
model_df['age_midpoint'] = model_df['age'].apply(age_to_midpoint)

for col in ['diag_1', 'diag_2', 'diag_3']:
    model_df[f'{col}_group'] = model_df[col].apply(diagnosis_group)

id_like_categories = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in id_like_categories:
    model_df[col] = model_df[col].astype('object')

drop_cols = [
    'encounter_id', 'patient_nbr', 'weight', 'payer_code',
    'diag_1', 'diag_2', 'diag_3', 'age', 'readmitted'
]

X = model_df.drop(columns=drop_cols + ['readmitted_binary'])
y = model_df['readmitted_binary']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric features:', numeric_features)
print('\nCategorical features:', categorical_features)
print('\nModeling shape:', X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print('Prepared train matrix:', X_train_prepared.shape)
print('Prepared test matrix:', X_test_prepared.shape)

In [ ]:
k_values = range(2, 9)
inertias = []
silhouette_scores = []

sample_size = min(10000, X_train_prepared.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(X_train_prepared.shape[0], size=sample_size, replace=False)
X_silhouette = X_train_prepared[sample_idx]

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    train_labels = kmeans.fit_predict(X_train_prepared)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_silhouette, train_labels[sample_idx]))

k_result = pd.DataFrame({
    'k': list(k_values),
    'inertia': inertias,
    'silhouette_score': silhouette_scores
})

display(k_result)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=k_result, x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set_title('Elbow Method')
sns.lineplot(data=k_result, x='k', y='silhouette_score', marker='o', ax=axes[1])
axes[1].set_title('Silhouette Score')
plt.tight_layout()
plt.show()

best_k = int(k_result.loc[k_result['silhouette_score'].idxmax(), 'k'])
print('Selected k:', best_k)

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
train_clusters = kmeans.fit_predict(X_train_prepared)
test_clusters = kmeans.predict(X_test_prepared)

train_profile = X_train.copy()
train_profile['cluster'] = train_clusters
train_profile['readmitted_binary'] = y_train.values

cluster_summary = train_profile.groupby('cluster').agg(
    patients=('readmitted_binary', 'size'),
    readmission_rate=('readmitted_binary', 'mean'),
    avg_time_in_hospital=('time_in_hospital', 'mean'),
    avg_lab_procedures=('num_lab_procedures', 'mean'),
    avg_medications=('num_medications', 'mean'),
    avg_inpatient_visits=('number_inpatient', 'mean'),
    avg_emergency_visits=('number_emergency', 'mean')
).sort_values('readmission_rate', ascending=False)

display(cluster_summary.round(3))

plt.figure(figsize=(8, 4))
sns.barplot(data=cluster_summary.reset_index(), x='cluster', y='readmission_rate')
plt.title('Readmission Rate by K-Means Cluster')
plt.xlabel('cluster')
plt.ylabel('readmission rate')
plt.show()

In [ ]:
cluster_risk = train_profile.groupby('cluster')['readmitted_binary'].mean()
baseline_rate = y_train.mean()

test_cluster_risk = pd.Series(test_clusters).map(cluster_risk).fillna(baseline_rate).values
y_pred = (test_cluster_risk >= baseline_rate).astype(int)

print('Baseline train readmission rate:', round(baseline_rate, 3))
print('\nConfusion matrix')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=['Actual No Readmission', 'Actual Readmission'],
    columns=['Predicted Low Risk', 'Predicted High Risk']
))

print('\nClassification report')
print(classification_report(y_test, y_pred, target_names=['No readmission', 'Readmission']))

In [ ]:
profile_columns = [
    'cluster', 'readmitted_binary', 'race', 'gender', 'age_midpoint',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'insulin', 'change',
    'diabetesMed', 'diag_1_group'
]

display(train_profile[profile_columns].head())

for cluster_id in cluster_summary.index:
    subset = train_profile[train_profile['cluster'] == cluster_id]
    print(f'\nCluster {cluster_id}')
    print('size:', len(subset))
    print('readmission rate:', round(subset['readmitted_binary'].mean(), 3))
    print('top diagnosis groups:')
    display(subset['diag_1_group'].value_counts(normalize=True).head(5).round(3))
    print('insulin distribution:')
    display(subset['insulin'].value_counts(normalize=True).round(3))

## Conclusion Guide

위 결과에서 `readmission_rate`가 높은 군집은 재입원 위험이 상대적으로 높은 환자군으로 해석할 수 있습니다. 특히 `number_inpatient`, `number_emergency`, `num_medications`, `time_in_hospital`, 진단군, 인슐린 처방 변화 같은 변수가 높은 군집의 특징을 확인하면 프로젝트 보고서에서 재입원 위험 요인을 설명하는 데 사용할 수 있습니다.

단, K-Means는 정답 라벨을 직접 학습하지 않는 비지도학습이므로 최종 예측 성능을 높이는 것이 목적이라면 Logistic Regression, Random Forest, XGBoost 같은 지도학습 분류 모델과 비교하는 것이 좋습니다.